**What is Schema Evolution?**
Schema evolution means your table can adapt to changes in the data structure—like adding new columns—without breaking.
In traditional systems, if your data schema changes (e.g., a new column is added), you might get errors. But Delta Lake supports automatic schema evolution.

**What is mergeSchema?**
mergeSchema is an option that tells Delta Lake to automatically update the table’s schema to match the new data being written.


Create Delta Table

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE tri.tri.employee_demo (
        emp_id INT,
        emp_name STRING,
        gender STRING,
        salary INT,
        Dept STRING
    )
    USING DELTA
    LOCATION '/Volumes/tri/tri/tri_files/tables/tri.tri.employee_demo'
    TBLPROPERTIES ('description' = 'table created for demo purpose')
""")

### creating target table

In [0]:
from delta.tables import *

DeltaTable.createOrReplace(spark) \
    .tableName("tri.tri.employee_demo") \
    .addColumn("emp_id", "INT") \
    .addColumn("emp_name", "STRING") \
    .addColumn("gender", "STRING") \
    .addColumn("salary", "INT") \
    .addColumn("Dept", "STRING") \
    .property("description", "table created for demo purpose") \
    .execute()

In [0]:
%sql
INSERT INTO tri.tri.employee_demo
VALUES (100, 'Stephen', 'M', 2000, 'IT');

In [0]:
%sql
SELECT * FROM tri.tri.employee_demo;

### Returns schema + table information.

In [0]:
%sql
DESCRIBE EXTENDED tri.tri.employee_demo;

### Returns Delta table metadata in a structured format (one row).

In [0]:
%sql
DESCRIBE DETAIL tri.tri.employee_demo;

### source -1
### Create DataFrame with a New Column (additionalcolumn2)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

employee_data = [
    (200, "Philipp", "M", 8000, "HR", "test data")
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("dept", StringType(), True),
    StructField("additionalcolumn1", StringType(), True)
])

df = spark.createDataFrame(
    data=employee_data,
    schema=employee_schema
)

display(df)

### writing source into taget with addition column

In [0]:
df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("tri.tri.employee_demo")

**error note**

[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 51766430-ef9e-4255-adba-b10fb61a6f5c).

### adding merge schema

In [0]:
df.write \
    .option("mergeSchema", "true") \
    .format("delta") \
    .mode("append") \
    .saveAsTable("tri.tri.employee_demo")

In [0]:
%sql
select * from tri.tri.employee_demo

### source-2
### Create DataFrame with a New Column (additionalcolumn2)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

employee_data = [
    (300, "David", "M", 8000, "HR", "dummy data")
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("dept", StringType(), True),
    StructField("additionalcolumn2", StringType(), True)
])

df = spark.createDataFrame(
    data=employee_data,
    schema=employee_schema
)

display(df)

In [0]:
df.write \
    .option("mergeSchema", "true") \
    .format("delta") \
    .mode("append") \
    .saveAsTable("tri.tri.employee_demo")

In [0]:
%sql
SELECT * FROM tri.tri.employee_demo;